# PDS 37 with VLTI/MATISSE: L-band binary fit

**MATISSE** observes in the thermal infrared. Here we use its L band (≈3–4 µm), where warm dust and gas
contribute more of the light than in GRAVITY's K band. All available epochs are fitted
together with the same two-disk binary model used for GRAVITY.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pmoired

sys.path.insert(0, str(Path.cwd().parent))  # make the local `pds37` package importable
from pds37 import config

plt.rcParams["figure.dpi"] = 80
from pds37 import oifits
from pds37.binary import binary_grid_search, image_window

LBAND_FILES = sorted(str(p) for p in config.MATISSE_LBAND_DIR.glob("*.fits"))
oifits.summarize_files(LBAND_FILES).drop(columns="telluric_corrected")

## 1. The data

In [ ]:
oi = pmoired.OI(LBAND_FILES, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI", "DPHI", "NFLUX"]})
oi.show()

The MATISSE pipeline can flag a sizeable part of the `OI_FLUX` spectrum. The fit below only uses `|V|`
and `T3PHI`, which are much less affected. `scripts/unflag_matisse_flux.py` writes copies of the files with
the flux flags cleared, for inspecting the full spectrum.

## 2. Binary grid search and refinement

Data binned ×5 in wavelength; companion searched within ±15 mas in 1.5 mas steps.

In [ ]:
oi = pmoired.OI(LBAND_FILES, binning=5, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI"]})

start = binary_grid_search(oi, max_sep=15, step=1.5)
oi.showGrid(interpolate=True, logV=1, cmap="gist_stern")

In [ ]:
oi.doFit(start, doNotFit=[], verbose=1)
oi.show(**image_window(oi.bestfit["best"]))